In [1]:
!pip install transformers==4.28.1 peft accelerate

In [2]:
import transformers
print(transformers.__version__)

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.28.1


In [3]:
from transformers import AutoTokenizer, AutoModel

model_name = "zhihan1996/DNABERT-2-117M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Model loaded successfully")

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\accelerate\u

Model loaded successfully


In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

cuda


In [5]:
sequence = "ACGTACGTACGT"

tokens = tokenizer(sequence, return_tensors="pt").to(device)

outputs = model(**tokens)

print(outputs[0].shape)

torch.Size([1, 7, 768])


In [6]:
import torch.nn as nn

class DNABERTClassifier(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs   # ✅ THIS FIXES THE ERROR
        )
        cls = outputs[0][:, 0, :]
        return self.classifier(cls)

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

model = DNABERTClassifier(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["Wqkv"],   # ✅ FIXED
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 294,912 || all params: 117,364,994 || trainable%: 0.2512776509833929


In [8]:
from torch.utils.data import Dataset

class DummyDataset(Dataset):
    def __len__(self):
        return 20
    
    def __getitem__(self, idx):
        seq = "ACGTACGTACGT"
        tokens = tokenizer(seq, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
        
        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(1)
        }

dataset = DummyDataset()

In [9]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=4)

model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for batch in loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids, attention_mask)
    loss = loss_fn(outputs, labels)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print("Loss:", loss.item())
    break

Loss: 0.7398563623428345


In [ ]:
!pip install pyfaidx

In [10]:
from pyfaidx import Fasta
genome = Fasta("D:\CFTR\Homo_sapiens_CFTR_sequence.fa") 

In [12]:
import pandas as pd

In [13]:
df = pd.read_csv(r"D:\CFTR\final_cftr_dataset.csv")
df.head()

,Variant cDNA name,variant determination,hgvs_genomic_grch38,chr,pos,ref,alt,cause,Name,Canonical SPDI,dbSNP ID,Variant type,Germline classification
0,c.1521_1523del,CF-causing,NC_000007.14:g.117559592_117559594del,chr7,117559590,ATCT,A,1,NaN,NaN,NaN,NaN,NaN
1,c.1624G>T,CF-causing,NC_000007.14:g.117587778G>T,chr7,117587778,G,T,1,NaN,NaN,NaN,NaN,NaN
2,c.1652G>A,CF-causing,NC_000007.14:g.117587806G>A,chr7,117587806,G,A,1,NaN,NaN,NaN,NaN,NaN
3,c.3909C>G,CF-causing,NC_000007.14:g.117652877C>G,chr7,117652877,C,G,1,NaN,NaN,NaN,NaN,NaN
4,c.3718-2477C>T,CF-causing,NC_000007.14:g.117639961C>T,chr7,117639961,C,T,1,NaN,NaN,NaN,NaN,NaN


In [15]:
df.shape

(1686, 13)

In [18]:
START = 117287120
END = 117715971

df[(df["pos"] >= START) & (df["pos"] <= END)].shape

(1686, 13)

In [20]:
df["chr"] = 7
df.head()

,Variant cDNA name,variant determination,hgvs_genomic_grch38,chr,pos,ref,alt,cause,Name,Canonical SPDI,dbSNP ID,Variant type,Germline classification
0,c.1521_1523del,CF-causing,NC_000007.14:g.117559592_117559594del,7,117559590,ATCT,A,1,NaN,NaN,NaN,NaN,NaN
1,c.1624G>T,CF-causing,NC_000007.14:g.117587778G>T,7,117587778,G,T,1,NaN,NaN,NaN,NaN,NaN
2,c.1652G>A,CF-causing,NC_000007.14:g.117587806G>A,7,117587806,G,A,1,NaN,NaN,NaN,NaN,NaN
3,c.3909C>G,CF-causing,NC_000007.14:g.117652877C>G,7,117652877,C,G,1,NaN,NaN,NaN,NaN,NaN
4,c.3718-2477C>T,CF-causing,NC_000007.14:g.117639961C>T,7,117639961,C,T,1,NaN,NaN,NaN,NaN,NaN


In [22]:
row = df.iloc[0]

chrom = str(row["chr"])
pos = row["pos"]
ref = row["ref"]
alt = row["alt"]

print(chrom, pos, ref, alt)

7 117559590 ATCT A


In [23]:
def get_local_pos(pos):
    return pos - START

In [24]:
def extract_window_local(pos, window=200):
    local_pos = pos - START
    
    start = local_pos - window
    end = local_pos + window
    
    seq = genome["7"][start:end].seq.upper()
    return seq

In [25]:
row = df.iloc[0]

pos = row["pos"]
ref = row["ref"]

seq = extract_window_local(pos)

center = 200

print("Genome:", seq[center:center+len(ref)])
print("Dataset:", ref)

Genome: ATCT
Dataset: ATCT


In [26]:
def apply_mutation(seq, ref, alt, window=200):
    center = window
    
    if seq[center:center+len(ref)] != ref:
        return None
    
    mutated = seq[:center] + alt + seq[center+len(ref):]
    return mutated

In [27]:
ref_seq = extract_window_local(pos)
mut_seq = apply_mutation(ref_seq, ref, alt)

In [28]:
center = 200

print("Before:", ref_seq[center-5:center+5])
print("After :", mut_seq[center-5:center+5])

Before: ATATCATCTT
After : ATATCATTGG


In [29]:
center = 200

print("Genome segment :", ref_seq[center:center+10])
print("Dataset ref    :", ref)
print("Dataset alt    :", alt)

Genome segment : ATCTTTGGTG
Dataset ref    : ATCT
Dataset alt    : A


In [30]:
print(ref_seq[center:center+len(ref)] == ref)

True
